# Experiment 1 — GPT-style Backbone (XGLM-564M)

This notebook trains and evaluates a **decoder-only GPT backbone** (facebook/xglm-564M)
on the 15-pair code-switching prediction task from SwitchLingua.

**Task:** Given a sequence of tokens from a bilingual utterance, predict at each position:
1. **Switch head** — will the next token be in a different language? (binary)
2. **Duration head** — if a switch occurs, how long is the burst? (3-class: small/medium/large)

**Key design choice:** XGLM is a decoder-only model with *native* causal masking —
position t can only attend to 0..t without any additional masking trick.

**Universality metric (σ):** We report the standard deviation of switch F1 across 15 language
pairs. A lower σ means the model generalises more uniformly across languages.

In [ ]:
import os, sys, gc, pickle
from pathlib import Path

# Point to the repo root so we can import the codeswitch library
REPO_ROOT = Path("__file__").resolve().parent.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from transformers import AutoTokenizer

from codeswitch.auth import hf_login
from codeswitch.config import (
    ALL_LANGUAGE_PAIRS, BACKBONE_MODEL_DEFAULTS, TRAIN_PAIRS, ZEROSHOT_PAIRS,
    DataConfig, ModelConfig, TrainConfig,
)
from codeswitch.data import build_datasets, make_collate_fn
from codeswitch.evaluate import evaluate, evaluate_per_pair, print_sigma_summary
from codeswitch.model import build_model
from codeswitch.results_json import save_results_json
from codeswitch.trainer import (
    compute_class_weights, make_warmup_cosine_scheduler, set_seed, train_epoch,
)
from codeswitch.visualize import (
    plot_grouped_f1_bars, plot_lr_schedule, plot_per_pair_training_curves,
    plot_training_history, plot_universality,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

## Configuration

All hyperparameters are set here. Change these to re-run the experiment with different settings.

In [ ]:
# ── Experiment config ─────────────────────────────────────────────────────────
model_cfg = ModelConfig(
    backbone="xglm",
    model_name="facebook/xglm-564M",
    max_len=256,
    dropout=0.1,
    freeze_encoder=False,
)

data_cfg = DataConfig(
    dataset_name="Shelton1013/SwitchLingua_text",
    max_samples_per_pair=6000,
    train_ratio=0.8,
)

train_cfg = TrainConfig(
    batch_size=32,
    num_epochs=16,
    base_lr=1e-5,
    head_lr_multiplier=50.0,   # head LR = 5e-4
    warmup_ratio=0.1,
    lambda_dur=1.0,            # multitask: switch + duration
    seed=42,
)

train_pairs    = TRAIN_PAIRS     # all 15 pairs
zeroshot_pairs = ZEROSHOT_PAIRS  # none held out

CHECKPOINT   = "checkpoints/best_xglm.pt"
RESULTS_PKL  = "results/train_xglm.pkl"
RESULTS_JSON = "results/train_xglm.json"
DATA_PICKLE  = "data/preprocessed.pkl"

print(f"Backbone:    {model_cfg.backbone}  ({model_cfg.model_name})")
print(f"Epochs:      {train_cfg.num_epochs}")
print(f"Base LR:     {train_cfg.base_lr}  |  Head LR: {train_cfg.head_lr}")
print(f"Warmup:      {train_cfg.warmup_ratio*100:.0f}% of total steps")
print(f"lambda_dur:  {train_cfg.lambda_dur}")
print(f"Train pairs: {len(train_pairs)}")

## Data Preprocessing

Load from cache if available, otherwise run the full LID + label-generation pipeline.

In [ ]:
if Path(DATA_PICKLE).exists():
    print(f"Loading cached data from {DATA_PICKLE}")
    with open(DATA_PICKLE, "rb") as f:
        all_stats = pickle.load(f)
    print(f"  Loaded {len(all_stats)} language pairs")
else:
    print("Cached data not found — running preprocessing (this may take a while)...")
    hf_login()   # reads HF_TOKEN env var or prompts

    from datasets import load_dataset
    from codeswitch.data import analyze_language_pair
    from codeswitch.lid import ProductionLID

    dataset = load_dataset(data_cfg.dataset_name)
    tokenizer_pre = AutoTokenizer.from_pretrained(model_cfg.model_name)
    lid = ProductionLID(device=0 if torch.cuda.is_available() else -1)

    all_stats = {}
    for lang1, lang2 in ALL_LANGUAGE_PAIRS:
        stats = analyze_language_pair(
            dataset["train"], lang1, lang2, lid, tokenizer_pre,
            max_samples=data_cfg.max_samples_per_pair,
        )
        if stats:
            all_stats[f"{lang1}-{lang2}"] = stats

    Path(DATA_PICKLE).parent.mkdir(parents=True, exist_ok=True)
    with open(DATA_PICKLE, "wb") as f:
        pickle.dump(all_stats, f)
    print(f"\n✓ Saved to {DATA_PICKLE}")

## Build Dataset and DataLoaders

In [ ]:
set_seed(train_cfg.seed)

tokenizer = AutoTokenizer.from_pretrained(model_cfg.model_name)
collate   = make_collate_fn(tokenizer.pad_token_id)

train_ds, val_ds = build_datasets(
    all_stats, train_pairs, tokenizer,
    train_ratio=data_cfg.train_ratio,
    max_len=model_cfg.max_len,
)
train_loader = DataLoader(
    train_ds, batch_size=train_cfg.batch_size,
    shuffle=True,  collate_fn=collate, num_workers=train_cfg.num_workers,
)
val_loader = DataLoader(
    val_ds, batch_size=train_cfg.batch_size,
    shuffle=False, collate_fn=collate, num_workers=train_cfg.num_workers,
)
print(f"Train batches: {len(train_loader)}  |  Val batches: {len(val_loader)}")

## Model, Optimizer, and Scheduler

In [ ]:
print("Computing inverse-frequency class weights...")
sw_w, dur_w = compute_class_weights(train_loader)
sw_w  = sw_w.to(device)
dur_w = dur_w.to(device)

sw_criterion  = nn.CrossEntropyLoss(weight=sw_w,  ignore_index=-100)
dur_criterion = nn.CrossEntropyLoss(weight=dur_w)

model = build_model(model_cfg).to(device)

optimizer = torch.optim.AdamW(
    [
        {"params": model.encoder.parameters(),       "lr": train_cfg.base_lr},
        {"params": model.switch_head.parameters(),   "lr": train_cfg.head_lr},
        {"params": model.duration_head.parameters(), "lr": train_cfg.head_lr},
    ],
    weight_decay=train_cfg.weight_decay,
)

total_steps  = train_cfg.num_epochs * len(train_loader)
warmup_steps = int(total_steps * train_cfg.warmup_ratio)
scheduler    = make_warmup_cosine_scheduler(optimizer, warmup_steps, total_steps)

plot_lr_schedule(
    warmup_steps, total_steps,
    base_lr=train_cfg.base_lr, head_lr=train_cfg.head_lr,
    output_path="results/xglm_lr_schedule.png",
)

print(f"\nTotal steps: {total_steps}  |  Warmup steps: {warmup_steps}")

## Training

Trains for `num_epochs` epochs. Saves the best checkpoint (highest validation switch F1).
Skip this cell if a checkpoint already exists.

In [ ]:
history:    list = []
best_sw_f1: float = 0.0
Path(CHECKPOINT).parent.mkdir(parents=True, exist_ok=True)

if Path(CHECKPOINT).exists():
    print(f"Checkpoint found at {CHECKPOINT} — loading (skip training).")
    model.load_state_dict(torch.load(CHECKPOINT, map_location=device))
else:
    for epoch in range(1, train_cfg.num_epochs + 1):
        losses  = train_epoch(model, train_loader, optimizer, device,
                              sw_criterion, dur_criterion, train_cfg, scheduler)
        metrics = evaluate(model, val_loader, device)
        pair_f1s = {
            k: v["switch_f1"]
            for k, v in evaluate_per_pair(
                model, all_stats, train_pairs, tokenizer, device,
                batch_size=train_cfg.batch_size,
                train_ratio=data_cfg.train_ratio,
                max_len=model_cfg.max_len,
            ).items()
        }
        history.append({
            "epoch":       epoch,
            "loss_sw":     losses["loss_sw"],
            "loss_dur":    losses["loss_dur"],
            "switch_f1":   metrics["switch_f1"],
            "duration_f1": metrics["duration_f1"],
            "pair_f1s":    pair_f1s,
        })
        print(
            f"Epoch {epoch:02d} | "
            f"loss_sw={losses['loss_sw']:.4f}  loss_dur={losses['loss_dur']:.4f} | "
            f"sw_F1={metrics['switch_f1']:.4f}  dur_F1={metrics['duration_f1']:.4f}"
        )
        if metrics["switch_f1"] > best_sw_f1:
            best_sw_f1 = metrics["switch_f1"]
            torch.save(model.state_dict(), CHECKPOINT)
            print(f"  ✓ New best: {best_sw_f1:.4f}")

print("Training complete.")

## Final Evaluation and Sigma-Universality

In [ ]:
model.load_state_dict(torch.load(CHECKPOINT, map_location=device))

eval_kwargs = dict(
    batch_size=train_cfg.batch_size,
    train_ratio=data_cfg.train_ratio,
    max_len=model_cfg.max_len,
)
final_train = evaluate_per_pair(model, all_stats, train_pairs,    tokenizer, device, **eval_kwargs)
final_zs    = evaluate_per_pair(model, all_stats, zeroshot_pairs, tokenizer, device, **eval_kwargs) \
              if zeroshot_pairs else {}

print_sigma_summary(final_train, final_zs)

## Visualizations

In [ ]:
if history:
    plot_training_history(history, output_path="results/xglm_training_history.png")

    pair_keys = [f"{l1}-{l2}" for l1, l2 in train_pairs]
    plot_per_pair_training_curves(
        history, pair_keys, output_path="results/xglm_per_pair_curves.png"
    )

plot_universality(
    final_train, final_zs,
    output_path="results/xglm_universality.png",
    title="XGLM-564M (GPT): Anticipatory F1 Across 15 Language Pairs",
)

plot_grouped_f1_bars(
    final_train,
    output_path="results/xglm_grouped_f1.png",
    title="XGLM-564M: Switch F1 vs Duration F1 per Language Pair",
)

## Save Results

In [ ]:
payload = {
    "backbone":         model_cfg.backbone,
    "model_name":       model_cfg.model_name,
    "history":          history,
    "train_results":    final_train,
    "zeroshot_results": final_zs,
}

Path(RESULTS_PKL).parent.mkdir(parents=True, exist_ok=True)
with open(RESULTS_PKL, "wb") as f:
    pickle.dump(payload, f)
print(f"✓ Results saved to {RESULTS_PKL}")

save_results_json(RESULTS_JSON, payload)
print(f"✓ JSON saved to {RESULTS_JSON}")